# VoiceForge presenter engine test (side by side)

Renders **your** line on **your** recording with each candidate lip-sync engine, on this free GPU, and shows the results next to each other with how long each took. Nothing leaves this session.

| Engine | Input | What it is |
|---|---|---|
| `latentsync15` | your recording | What Colab/Kaggle use today (LatentSync 1.5, 256 px), at the new 20-step / 1.5 settings |
| `musetalk15` | your recording | MuseTalk 1.5 (Tencent, MIT). Real-time class, ~4 GB |
| `echomimic3_flash` | a photo | EchoMimic V3 Flash (Ant Group, Apache 2.0). Image → talking presenter, ~12 GB |

**How to run**
1. Use a **separate** session from your VoiceForge notebook (this one needs the whole GPU). Kaggle: GPU T4 x2 or P100, Internet **On**. Colab: T4.
2. Fill in Cell 1, then **Run all**. The first run installs each engine (10–20 min each); the times shown separately for *install* and *render*.
3. Look at the side-by-side at the end. A zip of all results is saved (Kaggle: *Output* panel; Colab: it downloads).

LatentSync **1.6** (sharper, 512 px) only runs on Modal: to compare it, in VoiceForge switch **Presenter** off on the Colab and Kaggle cards (Admin → Backends), make the same clip in the studio, then switch them back on.

In [ ]:
# ===== Cell 1 · your files =====
# Each can be a URL (e.g. the link of your presenter on Admin -> Presenters)
# or a path in this session (Kaggle: /kaggle/input/<dataset>/file.mp4).
VIDEO = ""   # your presenter recording (the one the lip-sync redraws)
AUDIO = ""   # the narration: .wav/.mp3, OR any video with sound (e.g. a clip VoiceForge made). Empty = the sound of VIDEO
IMAGE = ""   # a front-facing photo for image -> presenter. Empty = a frame of VIDEO

ENGINES = ["latentsync15", "musetalk15", "echomimic3_flash"]
MAX_SECONDS = 12          # longer audio is cut to this, to keep the test short
DELETE_AFTER_EACH = True  # remove each engine's install after its run (saves ~20 GB of disk)

In [ ]:
# ===== Cell 2 · prepare the inputs =====
import os, sys, json, time, shutil, subprocess, urllib.request, glob, re
ROOT = "/tmp/bake"
IN = f"{ROOT}/in"; OUT = f"{ROOT}/out"
os.makedirs(IN, exist_ok=True); os.makedirs(OUT, exist_ok=True)
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle")

def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[-2500:])
    return r.stdout

def fetch(src, name):
    if not src:
        return None
    dest = f"{IN}/{name}"
    if src.startswith(("http://", "https://")):
        req = urllib.request.Request(src, headers={"User-Agent": "voiceforge-bakeoff"})
        with urllib.request.urlopen(req, timeout=300) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    else:
        shutil.copy(src, dest)
    return dest

if not VIDEO and ON_COLAB:
    from google.colab import files
    print("Choose your presenter recording:")
    up = files.upload(); VIDEO = "/content/" + next(iter(up))
assert VIDEO, "Set VIDEO in Cell 1 (a URL or a path)."

raw_video = fetch(VIDEO, "raw_video" + os.path.splitext(VIDEO.split("?")[0])[1][:5])
raw_audio = fetch(AUDIO, "raw_audio" + os.path.splitext(AUDIO.split("?")[0])[1][:5]) if AUDIO else raw_video
raw_image = fetch(IMAGE, "raw_image" + os.path.splitext(IMAGE.split("?")[0])[1][:5]) if IMAGE else None

def seconds(path):
    return float(sh(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", path]).strip())

# The narration: full quality for the final files, 16 kHz mono for the engines.
sh(["ffmpeg", "-y", "-v", "error", "-i", raw_audio, "-vn", "-t", str(MAX_SECONDS), "-ac", "2", "-ar", "44100", f"{IN}/audio_hq.wav"])
sh(["ffmpeg", "-y", "-v", "error", "-i", f"{IN}/audio_hq.wav", "-ac", "1", "-ar", "16000", f"{IN}/audio16k.wav"])
AUDIO_SECONDS = seconds(f"{IN}/audio_hq.wav")

# The recording: 25 fps (what the engines were trained on), no sound, as long
# as the narration, at most 1280 px.
sh(["ffmpeg", "-y", "-v", "error", "-stream_loop", "-1", "-i", raw_video, "-an", "-r", "25", "-t", f"{AUDIO_SECONDS + 0.4:.2f}",
    "-vf", "scale='if(gt(iw,ih),min(1280,iw),-2)':'if(gt(iw,ih),-2,min(1280,ih))'",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p", f"{IN}/video.mp4"])

# The photo: given, or a frame from the recording.
if raw_image:
    sh(["ffmpeg", "-y", "-v", "error", "-i", raw_image, "-frames:v", "1", f"{IN}/image.png"])
else:
    sh(["ffmpeg", "-y", "-v", "error", "-ss", "0.5", "-i", f"{IN}/video.mp4", "-frames:v", "1", f"{IN}/image.png"])

print(f"\nReady: {AUDIO_SECONDS:.1f}s of narration, recording and photo in {IN}")
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
print(sh("df -h /tmp | tail -1"))

In [ ]:
# ===== Cell 3 · the engines =====
RESULTS = {}

def gpu_env(index="0"):
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND")}
    env.update(CUDA_VISIBLE_DEVICES=index, PYTHONUNBUFFERED="1", MPLBACKEND="Agg", PIP_NO_CACHE_DIR="1", UV_NO_CACHE="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", HF_HUB_ENABLE_HF_TRANSFER="0")
    return env

def run_live(cmd, cwd=None, env=None, quiet_bars=True, timeout=60 * 90):
    # Show what the engine says as it works; keep the end for errors.
    print("$", " ".join(cmd)[:400], flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, env=env or gpu_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail, last_bar = [], 0.0
    deadline = time.time() + timeout
    for raw in iter(lambda: p.stdout.readline(), b""):
        line = raw.decode("utf-8", "replace").rstrip()
        tail = (tail + [line])[-60:]
        is_bar = "%|" in line or "it/s" in line or "s/it" in line
        if not is_bar or time.time() - last_bar > 15:
            print("   ", line[-240:], flush=True)
            if is_bar:
                last_bar = time.time()
        if time.time() > deadline:
            p.kill(); raise RuntimeError("timed out")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError("exit %s:\n%s" % (p.returncode, "\n".join(tail[-25:])))

def make_venv(path, python="3.10"):
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    if not os.path.exists(f"{path}/bin/python"):
        sh([sys.executable, "-m", "uv", "venv", "--seed", "--python", python, path])
    return f"{path}/bin/python"

def uv_install(py, *args):
    sh([sys.executable, "-m", "uv", "pip", "install", "--python", py, *args], env=gpu_env())

def hf_get(py, repo, dest, patterns=None, files=None, revision=None):
    code = (
        "import sys, json; from huggingface_hub import snapshot_download, hf_hub_download\n"
        f"repo, dest = {repo!r}, {dest!r}\n"
        f"files = {files!r}; patterns = {patterns!r}; rev = {revision!r}\n"
        "if files:\n"
        "    [hf_hub_download(repo, f, local_dir=dest, revision=rev) for f in files]\n"
        "else:\n"
        "    snapshot_download(repo, local_dir=dest, allow_patterns=patterns, revision=rev)\n"
    )
    sh([py, "-c", code], env=gpu_env())

def with_hq_audio(video, out):
    # Every result gets the same full-quality narration, so only the picture differs.
    sh(["ffmpeg", "-y", "-v", "error", "-i", video, "-i", f"{IN}/audio_hq.wav", "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k",
        "-shortest", "-movflags", "+faststart", out])
    return out


# ---------- LatentSync 1.5: exactly what VoiceForge runs on Colab/Kaggle ----------
def latentsync15():
    base = f"{ROOT}/latentsync15"
    os.environ["VOICEFORGE_PRESENTER_ROOT"] = f"{base}/engine"
    if not os.path.isdir(f"{base}/voiceforge"):
        sh(["git", "clone", "--depth", "1", "https://github.com/bijaybista006-art/voiceforge", f"{base}/voiceforge"])
    sys.path.insert(0, f"{base}/voiceforge")
    from backend import voiceforge_presenter as vp
    t0 = time.time()
    vp._install_engine()
    if vp.INSTALL["state"] != "ready":
        raise RuntimeError(vp.INSTALL["error"])
    install = time.time() - t0
    work = f"{base}/work"; os.makedirs(work, exist_ok=True)
    engine = vp.pick_engine(None, vp.installed_engines(), vp.PREFERRED_ENGINE)
    raw = f"{work}/out.mp4"
    cmd = vp.engine_command(engine, f"{IN}/video.mp4", f"{IN}/audio16k.wav", raw, vp.clamp_steps(None),
                            vp.clamp_guidance(None), 1247, f"{work}/tmp", runner=vp.write_runner(work))
    meter, said = vp.EngineProgress(), [0]
    def on_output(text):
        p = meter.feed(text)
        if p >= said[0] + 5:
            said[0] = p; print(f"    {p}% · {meter.describe()}", flush=True)
    t1 = time.time()
    env = vp._engine_env(); env["CUDA_VISIBLE_DEVICES"] = "0"
    proc = vp.stream_engine(cmd, vp.ENGINE_DIR, env, on_output)
    if proc.returncode != 0 or not os.path.exists(raw):
        raise RuntimeError(vp.describe_engine_failure(proc.returncode, proc.stderr, proc.stdout))
    render = time.time() - t1
    return with_hq_audio(raw, f"{OUT}/latentsync15.mp4"), install, render


# ---------- MuseTalk 1.5 ----------
def musetalk15():
    base = f"{ROOT}/musetalk15"; repo = f"{base}/repo"; m = f"{repo}/models"
    t0 = time.time()
    if not os.path.isdir(repo):
        sh(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk", repo])
    py = make_venv(f"{base}/venv", "3.10")
    marker = f"{base}/.ready"
    if not os.path.exists(marker):
        uv_install(py, "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2", "--index-url", "https://download.pytorch.org/whl/cu118")
        reqs = [l.strip() for l in open(f"{repo}/requirements.txt") if l.strip() and not l.startswith(("tensorflow", "tensorboard", "gradio"))]
        uv_install(py, *reqs, "huggingface_hub[cli]==0.30.2", "gdown", "openmim", "setuptools<70", "wheel")
        # mmpose pulls chumpy, which only builds without isolation.
        sh([py, "-m", "pip", "install", "-q", "--no-build-isolation", "chumpy==0.70"], env=gpu_env())
        sh([py, "-m", "mim", "install", "mmengine", "mmcv==2.0.1", "mmdet==3.1.0", "mmpose==1.1.0"], env=gpu_env())
        hf_get(py, "TMElyralab/MuseTalk", m, files=["musetalkV15/musetalk.json", "musetalkV15/unet.pth"])
        hf_get(py, "stabilityai/sd-vae-ft-mse", f"{m}/sd-vae", files=["config.json", "diffusion_pytorch_model.bin"])
        hf_get(py, "openai/whisper-tiny", f"{m}/whisper", files=["config.json", "pytorch_model.bin", "preprocessor_config.json"])
        hf_get(py, "yzd-v/DWPose", f"{m}/dwpose", files=["dw-ll_ucoco_384.pth"])
        os.makedirs(f"{m}/face-parse-bisent", exist_ok=True)
        sh([f"{base}/venv/bin/gdown", "--id", "154JgKpzCPW82qINcVieuPH3fZ2e0P812", "-O", f"{m}/face-parse-bisent/79999_iter.pth"])
        urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth", f"{m}/face-parse-bisent/resnet18-5c106cde.pth")
        open(marker, "w").close()
    install = time.time() - t0
    cfg = f"{base}/task.yaml"
    with open(cfg, "w") as f:
        f.write(f'task_0:\n  video_path: "{IN}/video.mp4"\n  audio_path: "{IN}/audio16k.wav"\n  result_name: "musetalk.mp4"\n')
    t1 = time.time()
    run_live([py, "-m", "scripts.inference", "--inference_config", cfg, "--result_dir", f"{base}/results",
              "--unet_model_path", "models/musetalkV15/unet.pth", "--unet_config", "models/musetalkV15/musetalk.json",
              "--version", "v15", "--use_float16", "--ffmpeg_path", os.path.dirname(shutil.which("ffmpeg"))], cwd=repo)
    render = time.time() - t1
    found = glob.glob(f"{base}/results/**/musetalk.mp4", recursive=True)
    if not found:
        raise RuntimeError("MuseTalk finished but wrote no video")
    return with_hq_audio(found[0], f"{OUT}/musetalk15.mp4"), install, render


# ---------- EchoMimic V3 Flash (photo -> presenter) ----------
def echomimic3_flash():
    base = f"{ROOT}/echomimic3"; repo = f"{base}/repo"; w = f"{base}/weights"
    t0 = time.time()
    if not os.path.isdir(repo):
        sh(["git", "clone", "--depth", "1", "https://github.com/antgroup/echomimic_v3", repo])
    py = make_venv(f"{base}/venv", "3.10")
    marker = f"{base}/.ready"
    if not os.path.exists(marker):
        uv_install(py, "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1", "--index-url", "https://download.pytorch.org/whl/cu121")
        reqs = [l.strip() for l in open(f"{repo}/requirements.txt") if l.strip() and not l.startswith("gradio")]
        uv_install(py, *reqs, "huggingface_hub")
        hf_get(py, "alibaba-pai/Wan2.1-Fun-V1.1-1.3B-InP", f"{w}/Wan2.1-Fun-V1.1-1.3B-InP")
        hf_get(py, "BadToBest/EchoMimicV3", f"{w}/hf", patterns=["echomimicv3-flash-pro/*"])
        if not glob.glob(f"{w}/hf/echomimicv3-flash-pro/**/chinese-wav2vec2-base", recursive=True):
            hf_get(py, "TencentGameMate/chinese-wav2vec2-base", f"{w}/chinese-wav2vec2-base")
        open(marker, "w").close()
    install = time.time() - t0
    transformer = glob.glob(f"{w}/hf/echomimicv3-flash-pro/**/diffusion_pytorch_model.safetensors", recursive=True)
    if not transformer:
        raise RuntimeError("Flash transformer weights not found in BadToBest/EchoMimicV3")
    wav2vec = (glob.glob(f"{w}/hf/echomimicv3-flash-pro/**/chinese-wav2vec2-base", recursive=True) or [f"{w}/chinese-wav2vec2-base"])[0]
    frames = int(AUDIO_SECONDS * 25)
    # T4/P100 have no fast bfloat16, so half precision; 512 px keeps it inside 16 GB.
    t1 = time.time()
    run_live([py, "infer_flash.py", "--image_path", f"{IN}/image.png", "--audio_path", f"{IN}/audio16k.wav",
              "--prompt", "A person is speaking to the camera, natural expression.", "--num_inference_steps", "8",
              "--config_path", "config/config.yaml", "--model_name", f"{w}/Wan2.1-Fun-V1.1-1.3B-InP",
              "--transformer_path", transformer[0], "--save_path", f"{base}/results", "--wav2vec_model_dir", wav2vec,
              "--sampler_name", "Flow_Unipc", "--video_length", str(frames), "--guidance_scale", "6.0",
              "--audio_guidance_scale", "3.0", "--audio_scale", "1.0", "--neg_scale", "1.0", "--neg_steps", "0",
              "--seed", "43", "--enable_teacache", "--teacache_threshold", "0.1", "--num_skip_start_steps", "5",
              "--riflex_k", "6", "--weight_dtype", "float16", "--sample_size", "512", "512", "--fps", "25", "--shift", "5.0"],
             cwd=repo)
    render = time.time() - t1
    found = glob.glob(f"{base}/results/*_output.mp4")
    if not found:
        raise RuntimeError("EchoMimic finished but wrote no video")
    return with_hq_audio(found[0], f"{OUT}/echomimic3_flash.mp4"), install, render

ENGINE_FUNCS = {"latentsync15": latentsync15, "musetalk15": musetalk15, "echomimic3_flash": echomimic3_flash}
print("Engines ready to run:", ", ".join(ENGINES))

In [ ]:
# ===== Cell 4 · run each engine (one fails? the others still run) =====
for name in ENGINES:
    print(f"\n========== {name} ==========", flush=True)
    t = time.time()
    try:
        path, install, render = ENGINE_FUNCS[name]()
        RESULTS[name] = {"ok": True, "file": path, "install_s": round(install), "render_s": round(render),
                         "per_second_of_audio": round(render / AUDIO_SECONDS, 1)}
        print(f"OK  {name}: render {render/60:.1f} min ({render/AUDIO_SECONDS:.1f}x the audio length), install {install/60:.1f} min")
    except Exception as e:
        RESULTS[name] = {"ok": False, "error": str(e)[-1500:], "after_s": round(time.time() - t)}
        print(f"FAILED {name}:\n{str(e)[-1500:]}")
    if DELETE_AFTER_EACH:
        shutil.rmtree(f"{ROOT}/{name}", ignore_errors=True)
        shutil.rmtree(f"{ROOT}/echomimic3" if name == "echomimic3_flash" else "/nonexistent", ignore_errors=True)
    print(sh("df -h /tmp | tail -1"))
print(json.dumps(RESULTS, indent=2))

In [ ]:
# ===== Cell 5 · side by side =====
import base64
from IPython.display import HTML, display

ok = [n for n in ENGINES if RESULTS.get(n, {}).get("ok")]
font = next((f for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                         "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"] if os.path.exists(f)), None)

def labelled(name):
    src = RESULTS[name]["file"]; dst = f"{OUT}/_lab_{name}.mp4"
    label = f"{name} - {RESULTS[name]['render_s']/60:.1f} min"
    vf = "scale=-2:540,pad=ceil(iw/2)*2:540"
    if font:
        vf += f",drawtext=fontfile={font}:text='{label}':x=12:y=12:fontsize=24:fontcolor=white:box=1:boxcolor=black@0.55:boxborderw=6"
    sh(["ffmpeg", "-y", "-v", "error", "-i", src, "-vf", vf, "-c:v", "libx264", "-crf", "18", "-an", dst])
    return dst

if ok:
    parts = [labelled(n) for n in ok]
    grid = f"{OUT}/side_by_side.mp4"
    if len(parts) > 1:
        inputs = sum([["-i", p] for p in parts], [])
        sh(["ffmpeg", "-y", "-v", "error", *inputs, "-i", f"{IN}/audio_hq.wav", "-filter_complex",
            "".join(f"[{i}:v]" for i in range(len(parts))) + f"hstack=inputs={len(parts)}[v]",
            "-map", "[v]", "-map", f"{len(parts)}:a", "-c:v", "libx264", "-crf", "20", "-c:a", "aac", "-shortest", grid])
    else:
        with_hq_audio(parts[0], grid)

    def player(path, title):
        data = base64.b64encode(open(path, "rb").read()).decode()
        return f"<div style='margin:8px 0'><b>{title}</b><br><video controls width='100%' style='max-width:1100px' src='data:video/mp4;base64,{data}'></video></div>"

    rows = "".join(
        f"<tr><td>{n}</td><td>{'yes' if r.get('ok') else 'failed'}</td><td>{(r.get('render_s') or 0)/60:.1f} min</td>"
        f"<td>{r.get('per_second_of_audio', '')}</td><td>{(r.get('install_s') or 0)/60:.1f} min</td></tr>"
        for n, r in RESULTS.items())
    display(HTML(
        f"<h3>Results for {AUDIO_SECONDS:.1f}s of narration</h3>"
        "<table><tr><th>engine</th><th>ok</th><th>render</th><th>seconds per second of audio</th><th>install</th></tr>"
        + rows + "</table>" + player(grid, "All together (same narration)")
        + "".join(player(RESULTS[n]["file"], n) for n in ok)))
else:
    print("No engine produced a video. The errors are above; send them to Claude.")

shutil.make_archive("/tmp/bakeoff_results", "zip", OUT)
with open(f"{OUT}/results.json", "w") as f:
    json.dump({"audio_seconds": AUDIO_SECONDS, "results": RESULTS}, f, indent=2)
if ON_KAGGLE:
    shutil.copy("/tmp/bakeoff_results.zip", "/kaggle/working/bakeoff_results.zip")
    print("Saved: /kaggle/working/bakeoff_results.zip (Output panel, right side)")
elif ON_COLAB:
    from google.colab import files
    files.download("/tmp/bakeoff_results.zip")
print(json.dumps({n: {k: v for k, v in r.items() if k != "file"} for n, r in RESULTS.items()}, indent=2))